In [3]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT=Path.cwd().parent
df=pd.read_csv(ROOT/'data'/'raw'/'Churn_Modelling.csv')

df=df.drop(columns=['RowNumber','CustomerId','Surname'])

print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df['Exited'].value_counts())
print(df['Exited'].value_counts(normalize=True)*100)

(10000, 11)
CreditScore          int64
Geography           object
Gender              object
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
dtype: object
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64
Exited
0    7963
1    2037
Name: count, dtype: int64
Exited
0    79.63
1    20.37
Name: proportion, dtype: float64


In [4]:
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.model_selection import train_test_split

df_model=df.copy()
df_model=pd.get_dummies(df_model,columns=['Geography'],drop_first=True)

le=LabelEncoder()
df_model['Gender']=le.fit_transform(df_model['Gender'])

print('Columns after encoding:')
print(df_model.columns.tolist())
print(df_model.shape)

Columns after encoding:
['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Geography_Germany', 'Geography_Spain']
(10000, 12)


In [5]:
#Balance to salary ratio
df_model['balance_salary_ratio']=(df_model['Balance']/df_model['EstimatedSalary'].replace(0,np.nan)).round(4)

df_model['age_group']=pd.cut(df_model['Age'],bins=[0,30,40,50,60,100],labels=['<30','30-40','40-50','50-60','60+']
)
df_model=pd.get_dummies(df_model,columns=['age_group'],drop_first=True)

#Products by Tenure
df_model['products_per_year']=(df_model['NumOfProducts']/df_model['Tenure'].replace(0,1).round(4))

df_model['zero_balance']=(df_model['Balance']==0).astype(int)

print('New features added:')
print(['balance_salary_ratio','age_group','products_per_year','zero_balance'])

New features added:
['balance_salary_ratio', 'age_group', 'products_per_year', 'zero_balance']


In [6]:
#Define features and target
X = df_model.drop(columns=['Exited'])
y = df_model['Exited']

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set:  {X_train.shape[0]:,} rows')
print(f'Test set:      {X_test.shape[0]:,} rows')
print(f'Train churn %: {y_train.mean()*100:.1f}%')
print(f'Test churn %:  {y_test.mean()*100:.1f}%')



Training set:  8,000 rows
Test set:      2,000 rows
Train churn %: 20.4%
Test churn %:  20.3%


In [7]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print('Before SMOTE:')
print(f'  Stayed (0):  {(y_train==0).sum():,}')
print(f'  Churned (1): {(y_train==1).sum():,}')
print('After SMOTE:')
print(f'  Stayed (0):  {(y_train_balanced==0).sum():,}')
print(f'  Churned (1): {(y_train_balanced==1).sum():,}')


Before SMOTE:
  Stayed (0):  6,370
  Churned (1): 1,630
After SMOTE:
  Stayed (0):  6,370
  Churned (1): 6,370


In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled  = scaler.transform(X_test)   # use train scaler on test!

print('Scaling complete')
print(f'X_train_scaled shape: {X_train_scaled.shape}')

# Save processed data
import joblib
joblib.dump(scaler, '../models/scaler.pkl')
print('Scaler saved to models/scaler.pkl')


Scaling complete
X_train_scaled shape: (12740, 18)
Scaler saved to models/scaler.pkl
